# Pipeline Logging Demo

This script demonstrates the user-facing logging added by #14. It requires a
running NDD/DataDesigner backend. Run cells individually in VS Code or PyCharm.

In [1]:
from anonymizer.logging import LoggingConfig, configure_logging  # noqa: F401

configure_logging()  # default: anonymizer INFO, DD suppressed
# configure_logging(LoggingConfig.verbose())  # anonymizer INFO + DD progress
# configure_logging(LoggingConfig.debug())    # everything DEBUG

/Users/amanoel/Code/Anonymizer/checkouts/feat-pipeline-logging/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import tempfile
from pathlib import Path

import pandas as pd

tmp_dir = tempfile.mkdtemp(prefix="logging_demo_")
csv_path = Path(tmp_dir) / "sample.csv"

pd.DataFrame(
    {
        "text": [
            "Alice Johnson works at Acme Corp in Portland, Oregon.",
            "Contact Bob Smith at bob.smith@example.com or 555-0123.",
            "Dr. Carol Lee treated patient #12345 on 2024-03-15.",
        ]
    }
).to_csv(csv_path, index=False)

print(f"Sample data saved to {csv_path}")

Sample data saved to /var/folders/66/lhpfpgl94z33ykf1njyb9y080000gp/T/logging_demo_iwjpjgfm/sample.csv


## Scenario 1: Full run with Redact replacement

In [3]:
from anonymizer.config.anonymizer_config import AnonymizerConfig, AnonymizerInput
from anonymizer.config.replace_strategies import Redact
from anonymizer.interface.anonymizer import Anonymizer

anonymizer = Anonymizer()
result = anonymizer.run(
    config=AnonymizerConfig(replace=Redact()),
    data=AnonymizerInput(source=str(csv_path)),
)
result.dataframe

[18:22:34] [INFO] 🔧 Anonymizer initialized with 3 model configs


[18:22:34] [INFO] 📂 Loaded 3 records from /var/folders/66/lhpfpgl94z33ykf1njyb9y080000gp/T/logging_demo_iwjpjgfm/sample.csv (column: 'text')


[18:22:34] [INFO] 🔍 Running entity detection on 3 records


[18:22:42] [INFO]   |-- ✅ Detection complete — 3 entities found across 3 records (0 failed)


[18:22:42] [INFO] 🔄 Running Redact replacement


[18:22:42] [INFO]   |-- ✅ Replacement complete (0 failed)


[18:22:42] [INFO] 🎉 Pipeline complete — 3 records processed, 0 total failures


,text,text_with_spans,final_entities,text_replaced
0,"Alice Johnson works at Acme Corp in Portland, ...",<first_name>Alice</first_name> <last_name>John...,"{'entities': [{'id': 'first_name_0_5', 'value'...",[REDACTED_FIRST_NAME] [REDACTED_LAST_NAME] wor...
1,Contact Bob Smith at bob.smith@example.com or ...,Contact <first_name>Bob</first_name> <last_nam...,"{'entities': [{'id': 'first_name_8_11', 'value...",Contact [REDACTED_FIRST_NAME] [REDACTED_LAST_N...
2,Dr. Carol Lee treated patient #12345 on 2024-0...,<occupation>Dr.</occupation> <first_name>Carol...,"{'entities': [{'id': 'occupation_0_3', 'value'...",[REDACTED_OCCUPATION] [REDACTED_FIRST_NAME] [R...


## Scenario 2: Preview mode

In [4]:
preview = anonymizer.preview(
    config=AnonymizerConfig(replace=Redact()),
    data=AnonymizerInput(source=str(csv_path)),
    num_records=2,
)
preview.dataframe

[18:22:42] [INFO] 📂 Loaded 3 records from /var/folders/66/lhpfpgl94z33ykf1njyb9y080000gp/T/logging_demo_iwjpjgfm/sample.csv (column: 'text')


[18:22:42] [INFO]   |-- 👀 Preview mode: processing 2 of 3 records


[18:22:42] [INFO] 🔍 Running entity detection on 3 records


[18:22:51] [INFO]   |-- ✅ Detection complete — 2 entities found across 2 records (0 failed)


[18:22:51] [INFO] 🔄 Running Redact replacement


[18:22:51] [INFO]   |-- ✅ Replacement complete (0 failed)


[18:22:51] [INFO] 🎉 Pipeline complete — 3 records processed, 0 total failures


,text,text_with_spans,final_entities,text_replaced
0,"Alice Johnson works at Acme Corp in Portland, ...",<first_name>Alice</first_name> <last_name>John...,"{'entities': [{'id': 'first_name_0_5', 'value'...",[REDACTED_FIRST_NAME] [REDACTED_LAST_NAME] wor...
1,Contact Bob Smith at bob.smith@example.com or ...,Contact <first_name>Bob</first_name> <last_nam...,"{'entities': [{'id': 'first_name_8_11', 'value...",Contact [REDACTED_FIRST_NAME] [REDACTED_LAST_N...


## Scenario 3: Detection only (no replacement)

In [5]:
from anonymizer.config.anonymizer_config import Rewrite

result_detect_only = anonymizer.run(
    config=AnonymizerConfig(rewrite=Rewrite()),
    data=AnonymizerInput(source=str(csv_path)),
)
result_detect_only.dataframe

[18:22:51] [INFO] 📂 Loaded 3 records from /var/folders/66/lhpfpgl94z33ykf1njyb9y080000gp/T/logging_demo_iwjpjgfm/sample.csv (column: 'text')


[18:22:51] [INFO] 🔍 Running entity detection on 3 records


[18:23:29] [INFO]   |-- ✅ Detection complete — 3 entities found across 3 records (0 failed)


[18:23:29] [INFO] 🎉 Pipeline complete — 3 records processed, 0 total failures


,text,text_with_spans,final_entities
0,"Alice Johnson works at Acme Corp in Portland, ...",<first_name>Alice</first_name> <last_name>John...,"{'entities': [{'id': 'first_name_0_5', 'value'..."
1,Contact Bob Smith at bob.smith@example.com or ...,Contact <first_name>Bob</first_name> <last_nam...,"{'entities': [{'id': 'first_name_8_11', 'value..."
2,Dr. Carol Lee treated patient #12345 on 2024-0...,<occupation>Dr.</occupation> <first_name>Carol...,"{'entities': [{'id': 'occupation_0_3', 'value'..."


## Cleanup

This file is temporary and should be removed before merging.